# Credit Risk in P2P Lending: A Visual Walkthrough

This notebook gives a visual overview of the project. It is meant to be read together with the presentation slides, but it can also stand on its own as a short guide to the main figures.

The project studies credit-risk prediction in peer-to-peer lending. A central theme is that predicting default is not only a machine-learning task. The definition of default, the prediction horizon, the country mix, and the use of predictions in decisions all change what the model output means.

## 1. Why Default Risk Is Not A Single Fixed Target

Before comparing models, we first need to decide what exactly counts as a default. This is less mechanical than it sounds: different definitions can lead to different measured default rates.

![Default label sensitivity](figures/default_label_sensitivity.png)

The figure shows that the estimated default rate changes when the default label is defined differently. This means the prediction target is partly a modeling choice, not just a fact waiting to be learned from the data.

Country differences add another layer. Loans from different countries do not necessarily come from the same risk distribution, so pooled averages can hide important structure.

![Country default rates](figures/country_default_rates.png)

The default-rate differences across countries suggest that a single global comparison may be too coarse. A model that looks reasonable on the full sample may behave differently within individual countries.

Default rates can also depend on when loans entered the platform. This matters because lending markets evolve over time.

![Entry vintage default rates](figures/entry_vintage_default_rates.png)

The comparison between early entry vintages and 2017 gives a simple view of time variation. If the population changes over time, then model evaluation is also affected by when the model is tested.

## 2. Model Probabilities And Calibration

A model can be useful for ranking loans without producing probabilities that are well calibrated. Calibration asks a direct question: among loans predicted to have a certain default probability, how many actually default?

![Calibration by model, 1-year horizon](figures/pod_calibration_1year.png)

For the one-year horizon, the diagonal line represents perfect calibration. Curves close to this line give predicted probabilities that match observed default frequencies more closely. The table on the right summarizes calibration error and Brier score for each model.

Changing the horizon changes the target. A one-year default probability and a lifetime default probability are not interchangeable, even if they are estimated from the same loan data.

![Calibration by model, lifetime horizon](figures/pod_calibration_lifetime.png)

The lifetime horizon has a different base rate and a different calibration pattern. This illustrates why the horizon should be stated explicitly whenever credit-risk probabilities are compared.

Brier score is one way to summarize probability forecasts, but it mixes several effects. Looking at its components helps separate calibration from other sources of predictive performance.

![Brier score decomposition](figures/brier_score_decomposition.png)

This decomposition helps explain why one number can be hard to interpret on its own. A model may improve one component while worsening another.

## 3. From Scores To Prediction Sets

The second part of the project studies conformal prediction. Instead of only outputting a score or probability, conformal methods produce prediction sets with a target coverage level.

![Split conformal coverage and width](figures/split_conformal_coverage_width.png)

The main trade-off is visible here: higher reliability often comes with wider or less decisive prediction sets. Coverage is valuable, but it has a cost.

Coverage can look acceptable on the full sample while still being uneven across groups. This is especially relevant when country composition is important.

![Group conditional coverage](figures/group_conditional_coverage.png)

The figure compares pooled and group-aware views of coverage. It is a reminder that a global guarantee may not describe every subgroup equally well.

## 4. What Happens When The Data Change?

Conformal prediction relies on assumptions about how calibration and test data are related. In lending data, those assumptions can be challenged by changes in country mix, time period, or label definition.

![Distribution shift diagnostic](figures/distribution_shift_diagnostic.png)

This diagnostic shows where the data may be shifting. If the test environment is not similar to the calibration environment, coverage statements become more fragile.

One response to time variation is to update the calibration procedure online, using recent errors to adjust future prediction sets.

![Online adaptation coverage](figures/online_adaptation_coverage.png)

Online adaptation can help track changing conditions, but it does not remove the underlying difficulty. The method still has to balance responsiveness and stability.

Label changes create a different kind of stress test. If the definition of default changes, then the target of prediction changes as well.

![Label robust coverage](figures/label_robustness_coverage.png)

This figure compares coverage under alternative default definitions. It shows that statistical guarantees should always be interpreted relative to the label being used.

## 5. From Prediction To Selection

In practice, credit-risk predictions are often used to select or reject loans. Once predictions become decisions, the statistical question changes again.

![Conformal selection FDR](figures/conformal_selection_fdr.png)

The target FDR level can be read as a risk-control dial. A stricter dial reduces false discoveries but also reduces the number of accepted opportunities.

The final comparison looks at native model uncertainty and conformal wrapping. A model's own uncertainty estimate is useful, but it is not automatically the same as a calibrated coverage guarantee.

![TabPFN conformal coverage](figures/tabpfn_conformal_coverage.png)

The comparison highlights the role of the conformal wrapper: it converts model scores into prediction sets aimed at a specified coverage level.

## How To Reproduce The Figures

The notebook is mainly for viewing figures. The computations are handled by the Python scripts in the repository.

```bash
# Generate the main figures
python bondora_credit_risk_analysis.py figures

# Generate only the calibration figures
python bondora_credit_risk_analysis.py figures --only calibration

# Generate the conformal figures after preparing the conformal data
python bondora_credit_risk_analysis.py build-master build-master
python conformal_experiments.py slides --master data/conformal_master.csv --preds data/step5_predictions.csv
```

The raw loan-level data are not included in the repository. The figures shown above are provided as static outputs so the notebook can be read directly on GitHub.